In [3]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import networkx as nx
from torch_geometric.data import Batch

# 尝试导入 forgi（可选）
try:
    import forgi.visual.mplotlib as fvm
    import forgi
    FORGI_AVAILABLE = True
except ImportError:
    FORGI_AVAILABLE = False
    print("Warning: forgi not available. Install with: pip install forgi")

# 导入项目相关模块
from model.main_model import RNA_ClassQuery_Model
from dataset.human import Mer100Dataset, one_hot_to_sequence

# 设置 Matplotlib 样式
plt.style.use('seaborn-v0_8-whitegrid')

print("所有库导入成功！")

所有库导入成功！


In [4]:
# 1. 加载配置
config_path = 'json/human.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# 2. 实例化数据集并获取单个样本
# 注意：我们这里只用于推理，所以只取训练集的一个样本即可
dataset = Mer100Dataset(mode='train', use_human3=True)
sample = dataset[0]  # 获取第一个样本
seq = one_hot_to_sequence(sample.x.numpy())
label = sample.y_site.numpy()

# 从 edge_index 构建邻接矩阵
edge_index = sample.edge_index.numpy()
num_nodes = sample.x.shape[0]
adj_matrix = np.zeros((num_nodes, num_nodes))
adj_matrix[edge_index[0], edge_index[1]] = 1

print(f"成功加载样本，序列长度: {len(seq)}")
print(f"邻接矩阵形状: {adj_matrix.shape}")
print(f"边数: {edge_index.shape[1]}")

# 3. 实例化模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNA_ClassQuery_Model(**config['model']).to(device)

# 4. 加载模型权重
checkpoint_path = 'logs/rna_classification_20260116_101325/checkpoints/epoch_030.pt'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

print(f"模型加载成功并设置为评估模式。")

使用内存映射加载human3数据: /home/dc/vscode/vscode20251230/human_and_plant/human3
数据集初始化完成 (mode=train):
  总样本数: 203993
  序列形状: (203993, 1001), dtype: |S1
  1001loc形状: (203993, 1001), dtype: int8
  12loc形状: (203993, 12), dtype: int8
  4loc形状: (203993, 4), dtype: int8
批量缓存文件不存在: /home/dc/vscode/vscode20260119/rgcnformer_sum/cache/human_train_structures_cache.npz
  提示: 请先调用 dataset.precompute_all_structures() 生成缓存
成功加载样本，序列长度: 1001
邻接矩阵形状: (1001, 1001)
边数: 2300


TypeError: unsupported operand type(s) for *: 'int' and 'dict'

In [ ]:
# 1. 准备模型输入
# 使用 PyG 的 Batch 功能来打包单个样本
batch = Batch.from_data_list([sample])
batch = batch.to(device)

# 2. 执行预测
with torch.no_grad():
    prediction_logits = model(batch.x, batch.edge_index, batch.batch)
    # 模型输出是 [batch_size, num_classes]
    # 我们需要对应"修饰"类别的概率, 这里假设类别1是修饰
    prediction_probs = torch.softmax(prediction_logits, dim=1)[0, 1]

# 3. 获取预测概率
predicted_prob = prediction_probs.item()

# 获取真实标签位置
true_site = np.where(label == 1)[0][0] if np.any(label == 1) else -1

print(f"模型预测完成。")
print(f"真实修饰位置: {true_site}")
print(f"预测修饰概率: {predicted_prob:.4f}")
print(f"注意：当前模型输出的是整体类别概率，不是逐位点的概率分布")

In [ ]:
# 创建一个示例的概率分布用于可视化
# 注意：由于模型输出的是整体类别概率而不是逐位点的概率，
# 这里我们创建一个模拟的逐位点概率分布用于演示可视化
print("注意：以下可视化使用模拟数据，展示可视化的效果")
prediction_probs_np = np.zeros(len(seq))
if true_site >= 0:
    prediction_probs_np[true_site] = 0.9  # 真实位点高概率
    # 在真实位点附近创建一些高概率区域
    half_window = 50
    start = max(0, true_site - half_window)
    end = min(len(seq), true_site + half_window)
    prediction_probs_np[start:end] = np.linspace(0.1, 0.8, end - start)
    prediction_probs_np[true_site] = 0.9  # 确保真实位点最高

plt.figure(figsize=(20, 6))
plt.plot(prediction_probs_np, label='Prediction Probability', color='blue', lw=2)
if true_site >= 0:
    plt.axvline(x=true_site, color='red', linestyle='--', label=f'True Site ({true_site})')

plt.title('RNA Modification Prediction (Matplotlib)', fontsize=16)
plt.xlabel('RNA Sequence Position', fontsize=12)
plt.ylabel('Modification Probability', fontsize=12)
plt.legend()
plt.xlim(0, len(seq))
plt.show()

In [ ]:
fig = go.Figure()

# 添加概率曲线
fig.add_trace(go.Scatter(
    x=np.arange(len(seq)),
    y=prediction_probs_np,
    mode='lines',
    name='Prediction Probability',
    hovertext=[f'Base: {seq[i]}<br>Pos: {i}<br>Prob: {p:.4f}' for i, p in enumerate(prediction_probs_np)],
    hoverinfo='text'
))

# 添加标记线
if true_site >= 0:
    fig.add_vline(x=true_site, line_width=2, line_dash="dash", line_color="red", name=f'True Site ({true_site})')

fig.update_layout(
    title='RNA Modification Prediction (Plotly)',
    xaxis_title='RNA Sequence Position',
    yaxis_title='Modification Probability',
    legend_title='Markers'
)

fig.show()

In [ ]:
# 创建图对象
G = nx.from_numpy_array(adj_matrix)

# 设置节点颜色
node_colors = ['#1f78b4'] * G.number_of_nodes()
if true_site >= 0 and true_site < G.number_of_nodes():
    node_colors[true_site] = 'red'

plt.figure(figsize=(12, 12))
pos = nx.kamada_kawai_layout(G)
nx.draw(G, pos, with_labels=False, node_size=20, node_color=node_colors, width=0.5)
title_text = f'GCN Graph Visualization (NetworkX) - Site {true_site} highlighted' if true_site >= 0 else 'GCN Graph Visualization (NetworkX)'
plt.title(title_text, fontsize=16)
plt.show()

In [ ]:
# 1. 从邻接矩阵提取碱基对
pairs = []
for i in range(adj_matrix.shape[0]):
    for j in range(i + 1, adj_matrix.shape[1]):
        if adj_matrix[i, j] == 1 and abs(i - j) > 3: # 假设距离大于3的连接为配对
            pairs.append((i, j))

print(f"找到 {len(pairs)} 个碱基对")

# 2. 如果 forgi 可用，则进行二级结构可视化
if FORGI_AVAILABLE:
    try:
        # 构建点括号表示的二级结构
        structure = ['.'] * len(seq)
        for i, j in pairs:
            structure[i] = '('
            structure[j] = ')'
        structure_str = ''.join(structure)
        
        # 从序列和结构字符串创建 BulgeGraph
        bg = forgi.BulgeGraph.from_dotbracket(structure_str)
        
        # 3. 定义颜色
        colors = {true_site + 1: 'red'} if true_site >= 0 else {}
        
        # 4. 绘图
        fig, ax = plt.subplots(figsize=(10, 10))
        fvm.plot_rna(bg, ax=ax)
        ax.set_title(f'RNA Secondary Structure (forgi) - Site {true_site} highlighted' if true_site >= 0 else 'RNA Secondary Structure (forgi)')
        plt.show()
    except Exception as e:
        print(f"注意：forgi 可视化失败: {e}")
        print("这可能是因为邻接矩阵不代表有效的二级结构，或者 forgi 版本/方法不兼容")
        print("建议：使用 LinearFold 生成二级结构，然后从结构字符串构建 BulgeGraph")
else:
    print("注意：forgi 未安装，跳过二级结构可视化")
    print("安装命令: pip install forgi")